project focuse more on backend engineering Data Integrity and Race Conditions.

tech stack :

FastAPI: The high-performance "front door" for requests.

PostgreSQL: Our source of truth. We will use Row-Level Locking (SELECT FOR UPDATE) to ensure that once a user starts a checkout, that specific stock item is "locked" until the transaction finishes.

SQLAlchemy (Async): To ensure our API doesn't hang while waiting for the database.

Pydantic: To validate that incoming order data is perfect.

In [47]:
!pip install fastapi uvicorn pyngrok sqlalchemy psycopg2-binary pydantic

# s1 : setting up the database

In [48]:
import os

# Create project folders
os.makedirs("app/database", exist_ok=True)
os.makedirs("app/models", exist_ok=True)

print("Project structure initialized.")

Project structure initialized.


In [49]:
%%writefile app/models/inventory.py
from sqlalchemy import Column , Integer ,String,Float ,CheckConstraint,DateTime
from sqlalchemy.sql import func
from sqlalchemy.ext.declarative import declarative_base

base = declarative_base()

class Product(base):
  __tablename__ ='products'

  id = Column(Integer,primary_key=True,index=True)
  sku = Column(String , unique=True,index=True)
  name = Column(String,index=True)
  price = Column(Float,index=True)
  stock_count = Column(Integer,default= 0)

  # Ensure stock never drops below zero at the DB level
  __table_args__= (
      CheckConstraint('stock_count >= 0',name='check_stock_not_negative'),
  )

class Order(base):
  __tablename__='Orders'

  id = Column(Integer , primary_key=True,index=True)
  product_id = Column(Integer)
  quantity = Column(Integer)
  status = Column(String,default='pending')
  created_at = Column(DateTime(timezone=True),server_default=func.now())

Overwriting app/models/inventory.py


In [50]:
#The Async Engine Setup
%%writefile app/database/config.py

Overwriting app/database/config.py


In [51]:
from sqlalchemy.ext.asyncio import create_async_engine ,AsyncSession
from sqlalchemy.orm import sessionmaker


In [52]:
DATABASE_URL = "sqlite+aiosqlite:///./inventory.db" #inproduction this changes to your server link

In [53]:
engine = create_async_engine(DATABASE_URL,echo=True)

In [54]:
Asynclocal_session = sessionmaker(engine,class_=AsyncSession,expire_on_commit=False)

In [55]:
%%writefile app/database/config.py
from sqlalchemy.ext.asyncio import create_async_engine ,AsyncSession
from sqlalchemy.orm import sessionmaker

DATABASE_URL = "sqlite+aiosqlite:///./inventory.db" #inproduction this changes to your server link

engine = create_async_engine(DATABASE_URL,echo=True)

Asynclocal_session = sessionmaker(engine,class_=AsyncSession,expire_on_commit=False)

async def get_db():
  async with Asynclocal_session() as session:
    yield session

async def init_db():
  from app.models.inventory import base
  async with engine.begin() as conn:
    # This creates the tables if they don't exist
    await conn.run_sync(base.metadata.create_all)

Overwriting app/database/config.py


In [56]:
import asyncio
import importlib
import app.database.config
import app.models.inventory

# Reload the module to ensure the latest changes are picked up
importlib.reload(app.database.config)
importlib.reload(app.models.inventory)

from app.database.config import init_db

# We use this to run async functions in a regular Colab cell
await init_db()
print("Database and tables created successfully!")

2026-05-05 10:33:11,807 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-05-05 10:33:11,811 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("products")


INFO:sqlalchemy.engine.Engine:PRAGMA main.table_info("products")


2026-05-05 10:33:11,814 INFO sqlalchemy.engine.Engine [raw sql] ()


...

2026-05-05 10:33:11,822 INFO sqlalchemy.engine.Engine [raw sql] ()


INFO:sqlalchemy.engine.Engine:[raw sql] ()


2026-05-05 10:33:11,825 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Database and tables created successfully!


# s2: setting up the service layer (brain )

In [57]:
os.makedirs('app/services',exist_ok=True)

In [58]:
# This service will use Row-Level Locking.

%%writefile app/services/inventory_service.py

from sqlalchemy.future import select
from sqlalchemy.ext.asyncio import AsyncSession
from app.models.inventory import Product, Order
from fastapi import HTTPException

Overwriting app/services/inventory_service.py


In [59]:
class InventoryService:
  @staticmethod
  async def process_order(db:AsyncSession,product_id:int,quantity:int):
    async with db.begin():# Start a transaction
    # 1. SELECT FOR UPDATE: Locks the row for this product
      query = select(Product).filter(Product.id == product_id).with_for_update()
      result = await db.execute(query)
      product = result.scalar_one_or_none()

      if not product:
        raise HTTPException(status_code=404,detail='prodcut not found')

      if product.stock_count < quantity:
        raise HTTPException(status_code = 400 ,detail='Insufficant stock')

      product.stock_count -= quantity

      new_order = Order(product_id=product_id,quantity=quantity)
      db.add(new_order)

      # Commit happens automatically when the 'async with' block ends
      return {"message": "Order successful", "remaining_stock": product.stock_count}


@staticmethod
async def get_all_products(db: AsyncSession):
    result = await db.execute(select(Product))
    return result.scalars().all()


# s3  Creating Pydantic Schemas

In [60]:
os.makedirs('app/schemas',exist_ok=True)

In [61]:
%%writefile app/schemas/inventory.py

from pydantic import BaseModel,Field
from typing import Optional

class OrderRequest(BaseModel):
  product_id : int
  qunatity : int = Field(gt=0,description='the quantity must be greater than 0')

class ProductRequests(BaseModel):
  id: int
  sku: str
  name: str
  price: float
  stock_count: int


  class config:
    from_attribute = True


Overwriting app/schemas/inventory.py


In [62]:
import importlib
import app.database.config

# Reload the config module to pick up any changes
importlib.reload(app.database.config)

from app.database.config import Asynclocal_session as AsyncSessionLocal
from app.models.inventory import Product

async def seed_data():
    async with AsyncSessionLocal() as db:
        async with db.begin():
            # Add a few "Hot Items"
            p1 = Product(sku="IPHONE-15", name="iPhone 15 Pro", price=999.99, stock_count=50)
            p2 = Product(sku="PS5-SLIM", name="PlayStation 5 Slim", price=499.99, stock_count=10)
            db.add_all([p1, p2])
        print("Database seeded with products!")

await seed_data()

2026-05-05 10:33:11,901 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-05-05 10:33:11,907 INFO sqlalchemy.engine.Engine INSERT INTO products (sku, name, price, stock_count) VALUES (?, ?, ?, ?) RETURNING id


INFO:sqlalchemy.engine.Engine:INSERT INTO products (sku, name, price, stock_count) VALUES (?, ?, ?, ?) RETURNING id


2026-05-05 10:33:11,908 INFO sqlalchemy.engine.Engine [generated in 0.00028s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('IPHONE-15', 'iPhone 15 Pro', 999.99, 50)


INFO:sqlalchemy.engine.Engine:[generated in 0.00028s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('IPHONE-15', 'iPhone 15 Pro', 999.99, 50)


2026-05-05 10:33:11,914 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: products.sku
[SQL: INSERT INTO products (sku, name, price, stock_count) VALUES (?, ?, ?, ?) RETURNING id]
[parameters: ('IPHONE-15', 'iPhone 15 Pro', 999.99, 50)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

# s4: fastapi setup

In [73]:
%%writefile app/main.py
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.ext.asyncio import AsyncSession
from app.database.config import get_db
from app.services.inventory_service import InventoryService
from app.schemas.inventory import OrderRequest, ProductRequests
from typing import List

app = FastAPI(title="High-Concurrency Inventory Engine")

@app.get("/", tags=["Root"])
async def read_root():
    return {"message": "Inventory Engine is Online", "docs": "/docs"}

@app.get("/products", response_model=List[ProductRequests], tags=["Inventory"])
async def get_products(db: AsyncSession = Depends(get_db)):
    return await InventoryService.get_all_products(db)

@app.post("/order", tags=["Transactions"])
async def place_order(order: OrderRequest, db: AsyncSession = Depends(get_db)):
    # This calls our high-concurrency logic with Row-Level Locking
    result = await InventoryService.process_order(
        db,
        product_id=order.product_id,
        quantity=order.qunatity
    )
    return result

Overwriting app/main.py


In [74]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn


NGROK_AUTH_TOKEN = "your auth token"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8080)
print(f" Public API URL: {public_url}")

nest_asyncio.apply()